In [ ]:
import os
from pathlib import Path

REVIEW_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "REVIEW_CODE_ROOT").is_file()),
    None,
)
if REVIEW_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted review package.")
os.chdir(REVIEW_ROOT)
REVIEW_FREQUENCY = os.environ.get("BFORTPROB_FREQUENCY", "daily")
if REVIEW_FREQUENCY not in {"daily", "weekly"}:
    raise ValueError("BFORTPROB_FREQUENCY must be daily or weekly.")


In [ ]:
import os
import sys
from dataclasses import dataclass, field
import warnings
from itertools import product
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
import json
import time

# =====================================================================
# Helper 
# =====================================================================
def select_arima_order(series, max_p=4, max_d=2, max_q=4):
    best_order = None
    best_aic = np.inf
    best_result = None
    
    candidate_orders = product(
        range(max_p + 1),
        range(max_d + 1),
        range(max_q + 1)
    )

    for order in candidate_orders:
        p, d, q = order
        
        # Avoid the degenerate ARIMA(0,0,0) unless it is genuinely useful.
        if p == 0 and d == 0 and q == 0:
            continue

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")

                model = ARIMA(
                    series,
                    order=order,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )

                result = model.fit()

            if result.aic < best_aic:
                best_aic = result.aic
                best_order = order
                best_result = result

        except Exception:
            continue

    if best_result is None:
        raise RuntimeError("ARIMA order search failed for all candidates")

    return best_order, best_result
    
# =====================================================================
# Multistep forecasting: training, testing 
# =====================================================================
def trainAndtest(trainf: str, testf: str, look_back: int, horizon: int, slide: int,
                 outf:str, txt_outf: str):
    # ---------------------------------------------------------------
    # Identify supervised columns
    # ---------------------------------------------------------------

    if not os.path.exists(trainf):
        raise ValueError(trainf+" does not exist")
    if not os.path.exists(testf):
        raise ValueError(testf+" does not exist")

    train_df = pd.read_csv(trainf)
    test_df = pd.read_csv(testf)
    print("# Raw shapes of train, test:", train_df.shape, test_df.shape)

    # Column groupings
    x_cols = [f"X_{i}" for i in range(look_back)]
    y_cols = [f"Y_{i}" for i in range(horizon)]

    trainX = train_df[x_cols]
    trainY = train_df[y_cols]
    testX  = test_df[x_cols]
    testY  = test_df[y_cols]

    print("# Supervised shapes of train X Y and test X Y:", trainX.shape, trainY.shape,
          testX.shape, testY.shape)

    # ---------------------------------------------------------------
    # Reconstruct a chronological training series
    # ---------------------------------------------------------------
    # Example for slide=1:
    # row 0 X = t0 ... t249
    # row 1 X = t1 ... t250
    # Therefore, after the first row, each next row contributes only
    # its last `slide` new observations.
    #
    # Finally, append the final trainY row because it is still part of
    # the training period and gives the known future after the final X.
    # ---------------------------------------------------------------
    train_series = list(trainX.iloc[0].values)

    for i in range(1, len(trainX)):
        new_values = trainX.iloc[i].values[-slide:]
        train_series.extend(new_values)

    train_series.extend(trainY.iloc[-1].values)

    train_series = pd.Series(train_series).astype(float)

    print("# Reconstructed training series length:", len(train_series))

    # ---------------------------------------------------------------
    # Fit a StandardScaler on the train series
    # ---------------------------------------------------------------

    scaler = StandardScaler()
    
    train_series_std = scaler.fit_transform(
        train_series.to_numpy().reshape(-1, 1)
    ).ravel()
    
    train_series_std = pd.Series(
        train_series_std,
        dtype=float
    )

    # ---------------------------------------------------------------
    # Select ARIMA order using a small AIC search
    # ---------------------------------------------------------------
    # This avoids hard-coding an ARIMA order.
    # The grid is intentionally modest because ARIMA fitting can be slow.
    # ---------------------------------------------------------------
    print("Selecting ARIMA order...")
    train_start_time = time.perf_counter()
    best_order, fitted_model = select_arima_order(train_series_std)
    train_end_time = time.perf_counter()
    training_time_seconds = train_end_time - train_start_time

    print("Selected ARIMA order:", best_order)
    print("Training AIC:", fitted_model.aic)
    print("Training time seconds:", training_time_seconds)

    # ---------------------------------------------------------------
    # 5. Forecast each test row
    # ---------------------------------------------------------------
    # Each testX row is a known look-back window. We apply the trained
    # ARIMA parameters to that history and forecast the next `horizon`
    # steps.
    #
    # This avoids refitting on testY and avoids leakage.
    # ---------------------------------------------------------------
    test_start_time = time.perf_counter()
    preds = []
    
    for i in range(len(testX)):

        # Transform history using the same trained StandardScaler parameters.
        history = testX.iloc[i].to_numpy(
            dtype=float
        )
        
        history_std = scaler.transform(
            history.reshape(-1, 1)
        ).ravel()
        
        history_std = pd.Series(
            history_std,
            dtype=float
        )

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")

                # Reuse trained parameters on the new observed scaled history.
                applied_result = fitted_model.apply(history_std, refit=False)
                forecast_std = applied_result.forecast(steps=horizon)

                # Inverse-transform forecasts back to original scale.
                forecast_std = np.asarray(
                    forecast_std,
                    dtype=float
                )
                
                forecast = scaler.inverse_transform(
                    forecast_std.reshape(-1, 1)
                ).ravel()

        except Exception as e:
            raise RuntimeError(f"Forecasting failed for test row {i}: {e}")
            """
            # Fallback: if `apply` is unavailable or fails in a statsmodels
            # version, fit the selected ARIMA structure on the test history.
            # This is slower, but still uses only testX, not testY.
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")

                temp_model = ARIMA(
                    history,
                    order=best_order,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )

                temp_result = temp_model.fit()
                forecast = temp_result.forecast(steps=horizon)
            """
        preds.append(np.asarray(forecast))

    pred_arr = np.vstack(preds)
    true_arr = testY.values

    test_end_time = time.perf_counter()
    testing_time_seconds = test_end_time - test_start_time

    # ---------------------------------------------------------------
    # Compute metrics
    # ---------------------------------------------------------------
    overall_mae = mean_absolute_error(true_arr.ravel(), pred_arr.ravel())
    overall_rmse = np.sqrt(mean_squared_error(true_arr.ravel(), pred_arr.ravel()))

    horizon_mae = []
    horizon_rmse = []

    for h in range(horizon):
        h_mae = mean_absolute_error(true_arr[:, h], pred_arr[:, h])
        h_rmse = np.sqrt(mean_squared_error(true_arr[:, h], pred_arr[:, h]))

        horizon_mae.append(h_mae)
        horizon_rmse.append(h_rmse)

    print("Overall MAE :", overall_mae)
    print("Overall RMSE:", overall_rmse)

    # ---------------------------------------------------------------
    # Save predictions and results
    # ---------------------------------------------------------------
    out_df = testX.copy()

    for i, col in enumerate(y_cols):
        out_df[col] = pred_arr[:, i]

    out_df.to_csv(outf, index=False)
    print(f"Saved forecasts to: {outf}")

    results_to_save = {
        "order": list(best_order),
        "train_aic": float(fitted_model.aic),
        # Save scaler paremeters as well.
        "scaler_mean": float(scaler.mean_[0]),
        "scaler_scale": float(scaler.scale_[0]),
        "training_time_seconds": float(training_time_seconds),
        "testing_time_seconds": float(testing_time_seconds),
        "overall_mae": float(overall_mae),
        "overall_rmse": float(overall_rmse),
        "horizon_mae": [float(v) for v in horizon_mae],
        "horizon_rmse": [float(v) for v in horizon_rmse],
    }

    with open(txt_outf, "w") as f:
        json.dump(results_to_save, f, indent=4)

    print(f"Saved forecasts to: {outf}")
    print(f"Saved full results to: {txt_outf}")
    print("Testing time seconds:", testing_time_seconds)

    return

In [ ]:
if __name__ == "__main__":

    from pathlib import Path

    # ============================================================
    # SETTINGS
    # ============================================================

    look_back = 250
    horizon = 20
    slide = 1
    model = "arima"

    # Folder containing 80/20 split train/test CSVs
    source_dir = Path(f"BForTFin_log_return/arima_split_datasets/output_{REVIEW_FREQUENCY}")

    # Where ARIMA forecasts will be saved
    save_dir = Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}/arima_standardized")

    # Where ARIMA metadata/results JSON files will be saved
    rslt_dir = save_dir / "results"

    save_dir.mkdir(parents=True, exist_ok=True)
    rslt_dir.mkdir(parents=True, exist_ok=True)

    asset_classes = [
        "G7",
        "EM",
        "MSCI",
        "Rates",
        "REIT",
        "Equities",
        "ETF",
        "Crypto",
    ]

    # ============================================================
    # RUN ARIMA
    # ============================================================

    for asset_class in asset_classes:

        asset_dir = source_dir / asset_class

        if not asset_dir.exists():
            print(f"Missing folder: {asset_dir}")
            continue

        output_asset_dir = save_dir / asset_class
        results_asset_dir = rslt_dir / asset_class

        output_asset_dir.mkdir(parents=True, exist_ok=True)
        results_asset_dir.mkdir(parents=True, exist_ok=True)

        # Find all training files
        train_files = sorted(
            asset_dir.glob(
                "train_*_no_preprocess_"
                "lookback-250_horizon-20_slide-1.csv"
            )
        )

        print(
            f"\n===== {asset_class}: "
            f"{len(train_files)} datasets ====="
        )

        for train_csv in train_files:

            suffix = (
                "_no_preprocess_"
                "lookback-250_"
                "horizon-20_"
                "slide-1.csv"
            )

            dataset_name = (
                train_csv.name
                .removeprefix("train_")
                .removesuffix(suffix)
            )

            # Skip Tether
            if "tether" in dataset_name.lower():
                print(f"Skipping Tether: {dataset_name}")
                continue

            test_csv = asset_dir / (
                f"test_{dataset_name}"
                f"_no_preprocess_"
                f"lookback-250_"
                f"horizon-20_"
                f"slide-1.csv"
            )

            if not test_csv.exists():
                print(f"Missing test file: {test_csv}")
                continue

            forecasted_csv = output_asset_dir / (
                f"test_{dataset_name}"
                f"_no_preprocess_"
                f"lookback-250_"
                f"horizon-20_"
                f"slide-1_"
                f"{model}.csv"
            )

            results_txt = results_asset_dir / (
                f"test_{dataset_name}"
                f"_no_preprocess_"
                f"lookback-250_"
                f"horizon-20_"
                f"slide-1_"
                f"{model}.json"
            )

            print("\n----------------------------------------")
            print("Dataset:", dataset_name)
            print("Train:", train_csv)
            print("Test :", test_csv)
            print("Output:", forecasted_csv)

            trainAndtest(
                str(train_csv),
                str(test_csv),
                look_back,
                horizon,
                slide,
                str(forecasted_csv),
                str(results_txt),
            )